# Section 1: Problem and Population

The issue in question that we are going over is regarding homelessness. These homeless individuals face the hardship of high barrier to entry for jobs, employment layoffs, and high living/housing costs that keep these people in poverty and homelessness. Alongside this, by not having specific actions (specifically by AI) to not only remedy any issues caused by homeless individuals but also remedy them from being in poverty, it makes it hard for this cycle of poverty into issues to stop. This connects to the UN's first SDG being "No poverty". Pertaining to M1 the biggest change is more so not necessarily targetting how to solve this issue as a whole, but modifying AI systems to be able to recognize situations related to this and creating action plans accordingly through distinguishing the exact issues and not confusing it with other issues instead of trying to solve a huge  global issue with many differing solutions as it would be more feasible to make steps towards this one at a time.

# Section 2

INPUT : Individuals or homeowners reporting to San Jose's 311 system within an added "homelessness" branch about an issue pertaining to or related to homeless individuals

AI Processing: The AI system derives info from the call/message according to this classification : (location, problem caused, urgency, department to be called, and resident language)

Output: From this, the AI system contacts a human and tells them a brief overview of the situation, the info they have derived, and its suggestions as to what departments to call alongside what possible actions should be taken such as who else to contact.

Real-World Action : From this, the department comes and takes action using their experience, critical-thinking skills, and AI reccomendations in mind to be able to provide by contacting the right individuals which are suggested by AI in order to provide the best remedy to the situation taking extra preventative measures that would help in the long-term rather than focus on just solving the short-term issue.



# Section 3

In [1]:
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata
import json
import time
from collections import Counter


genai.configure(api_key=userdata.get('GEMINI_API_KEY'))


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
resident_message = """
I am calling about an issue near the Guadalupe River Trail by W San Carlos Street.
There are several tents close to the walking path, and some trash is starting to build up near the sidewalk.
The path is still partly usable, but it is becoming harder for pedestrians to pass safely.
I am not sure which department should handle this, but I think the city should check on both the cleanup issue
and whether the people staying there may need outreach support.
"""

Import needed extensions + call message prompt

In [3]:
schema_prompt = """
Extract information from this San Jose 311 resident complaint.

Return ONLY valid JSON with exactly these six fields:

{
  "location": string,
  "reported_issue": string,
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "primary_responder": string,
  "supporting_services": list of strings,
  "resident_language": string
}

Field guide:
- location: the street address, park, intersection, or location described
- reported_issue: the civic, safety, environmental, or service-related issue described
- urgency: LOW, MEDIUM, or HIGH
- primary_responder: the first human department or service that should review/respond first
- supporting_services: other services the primary responder may consider contacting, such as "Homeless Outreach Services", "Emergency Medical Services", "Environmental Services", "Public Works", "Code Enforcement", or "Parks & Recreation"
- resident_language: language the resident wrote in, such as "English", "Spanish", or "Vietnamese"

Urgency guide:
LOW = minor litter, general concern, or non-urgent service request
MEDIUM = ongoing issue, blocked access, large debris, or situation requiring city follow-up
HIGH = hazardous materials, immediate safety risk, medical risk, fire risk, or violence

Do not blame people experiencing homelessness unless the report explicitly provides evidence.
The AI should not directly provide treatment, enforcement, or final decisions. It should identify the primary human responder and possible supporting services for human review.
No explanation. No markdown. JSON only.
"""

This prompt is used to provide the classification for AI when it derives and reads through the call in order to get the exact info that is needed but also to be able to achieve and maintain consistency,

In [4]:

def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )

    response = m.generate_content(message)
    time.sleep(12)  # stays under free tier rate limit

    raw = response.text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return json.loads(raw)


def extract_consistent(message):
    results = []

    # Run extraction 3 times internally
    for _ in range(3):
        try:
            result = extract_structured(message)
            results.append(result)
        except Exception:
            continue

    if not results:
        return {"error": "No valid JSON outputs were returned."}

    expected_keys = [
        "location",
        "reported_issue",
        "urgency",
        "primary_responder",
        "supporting_services",
        "resident_language"
    ]

    final = {}

    for key in expected_keys:
        values = [r.get(key, "Unknown") for r in results]

        if key == "supporting_services":
            flat_services = []

            for value in values:
                if isinstance(value, list):
                    flat_services.extend(value)
                else:
                    flat_services.append(value)

            # Remove duplicates while keeping order
            final[key] = list(dict.fromkeys(flat_services))

        else:
            final[key] = Counter(values).most_common(1)[0][0]

    return final


final_result = extract_consistent(resident_message)
print(json.dumps(final_result, indent=2, ensure_ascii=False))

{
  "location": "Guadalupe River Trail by W San Carlos Street",
  "reported_issue": "Several tents close to the walking path and trash buildup near the sidewalk, making it harder for pedestrians to pass safely.",
  "urgency": "MEDIUM",
  "primary_responder": "Homeless Outreach Services",
  "supporting_services": [
    "Public Works",
    "Parks & Recreation"
  ],
  "resident_language": "English"
}


This runs the extraction 3 times to be able to ensure consistency and eliminate hallucinations as given the emergency aspect of these situations, it will be needed to ensure no hallucinations or mistake occur

In [5]:
civic_questions = [
    (
        "PROBLEM",
        "Describe the civic, safety, environmental, or service-related issue in this 311 call/message. Be specific and avoid blaming people unless there is clear evidence."
    ),
    (
        "IMPACT",
        "What are the public health, safety, service, or community impacts described or reasonably implied in this 311 call/message?"
    ),
    (
        "URGENCY",
        "Rate the urgency as LOW, MEDIUM, or HIGH. Explain your rating in 2–3 sentences based only on the call/message."
    ),
    (
        "ACTION",
        "Which San Jose city department or outreach service should review this? What action and precautions should they take?"
    )
]

def analyze_call(message, question):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash"
    )

    prompt = f"""
You are analyzing a San Jose 311 call/message.

Resident call/message:
{message}

Question:
{question}
"""

    response = m.generate_content(prompt)
    time.sleep(12)

    return response.text.strip()


def analyze_call_consistent(message, question, n=3):
    answers = []

    for _ in range(n):
        answer = analyze_call(message, question)
        answers.append(answer)

    synthesized_answer = Counter(answers).most_common(1)[0][0]

    return synthesized_answer


civic_results = {"answers": {}}

for label, question in civic_questions:
    print(f"--- {label} ---")

    answer = analyze_call_consistent(resident_message, question, n=3)

    civic_results["answers"][label] = answer

    print(answer)
    print()

--- PROBLEM ---
This 311 call describes a multi-faceted issue near the Guadalupe River Trail by W San Carlos Street:

1.  **Safety/Civic:** Several tents are close to the walking path, making it "harder for pedestrians to pass safely" and impeding the full and safe usability of a public trail.
2.  **Environmental/Civic:** There is "trash starting to build up near the sidewalk," indicating a litter and waste management problem in a public area.
3.  **Service-related:** The resident explicitly requests addressing the "cleanup issue" and suggests the city "check on... whether the people staying there may need outreach support," highlighting a need for both waste management services and social support services for individuals experiencing homelessness.

--- IMPACT ---
Based on the resident's call, here are the public health, safety, service, and community impacts:

**Public Health Impacts:**

*   **Pest Infestation Risk:** Accumulation of trash can attract rodents (rats, mice) and insects 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2854.41ms


**MEDIUM**

The path is still partly usable, but the accumulation of tents and trash is making it increasingly difficult and less safe for pedestrians. While not an immediate impassable obstruction, the situation is clearly deteriorating and poses a growing safety concern for public access.

--- ACTION ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3083.49ms


Based on the resident's call, several San Jose city departments and services should review this issue collaboratively due to its multi-faceted nature involving homelessness, public safety, and environmental concerns on a public trail.

## San Jose City Departments/Outreach Services to Review:

1.  **San Jose Homelessness Response Team (HRT) / Housing Department:**
    *   **Reason:** Direct request for "outreach support" for people staying in tents, and addressing the root causes of homelessness. The HRT is specifically designed for addressing encampments and connecting individuals with services.

2.  **Parks, Recreation and Neighborhood Services (PRNS):**
    *   **Reason:** The issue is on the "Guadalupe River Trail," which falls under PRNS jurisdiction for maintenance, safety, and management of parks and trails. They are responsible for ensuring the trail is safe and accessible.

3.  **Environmental Services Department (ESD):**
    *   **Reason:** The resident explicitly mentions "t

This part serves to analyze the situation through four ways concising what it had derived initially in order to prepare itself from creating a response.

In [6]:
multilingual_prompt_active = """
You are a helpful assistant for the San Jose 311 civic reporting system.

A resident has submitted a 311 report. Write a brief, polite acknowledgment that:
1. Confirms the report was received.
2. Names the reported issue.
3. Explains that the report will be reviewed by a human dispatcher or city staff member.
4. States that possible supporting services may be considered, such as outreach, sanitation, public works, or emergency services if needed.
5. Gives an expected response timeframe of 3–5 business days for non-emergency issues.

Important:
- Do not promise that a specific department will respond.
- Do not provide treatment, enforcement, or final decisions.
- Do not blame people experiencing homelessness.
- If the report describes immediate danger, tell the resident to contact emergency services.
- Keep the response under 80 words.
- Detect the resident's language and respond in that same language.
"""

def generate_resident_response(message, structured_result):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=multilingual_prompt_active
    )

    prompt = f"""
Resident message:
{message}

Structured report:
{json.dumps(structured_result, indent=2, ensure_ascii=False)}
"""

    response = m.generate_content(prompt)
    time.sleep(12)

    return response.text.strip()


resident_reply = generate_resident_response(resident_message, final_result)

print("--- Resident Acknowledgment ---")
print(resident_reply)

--- Resident Acknowledgment ---
Thank you for your report regarding several tents and trash buildup near the Guadalupe River Trail. Your submission has been received and will be reviewed by a human dispatcher or city staff. We will consider possible supporting services, such as outreach, sanitation, or public works. For non-emergency issues, you can expect a response within 3-5 business days.


This part creates the the framework for how the AI should response to ensure consistency, and it also prints the output, being the response.

# Section 4: Edge Case Elicitation

In [ ]:
edge_case_message = """
There are two people sitting near the park entrance with bags next to them.
They might be homeless, but I am not sure. They are not blocking anyone, and I do not see trash or danger.
The area just feels uncomfortable.
"""

print("=== STRUCTURED OUTPUT ===")
edge_result = extract_structured(edge_case_message)
print(json.dumps(edge_result, indent=2, ensure_ascii=False))
print()

print("=== CIVIC ANALYSIS ===")
for label, question in civic_questions:
    print(f"--- {label} ---")
    try:
        answer = analyze_call(edge_case_message, question)
        print(answer)
    except Exception as e:
        print("Skipped because the Gemini API quota/rate limit was reached.")
    print()

print("=== RESIDENT RESPONSE ===")
try:
    edge_reply = generate_resident_response(edge_case_message, edge_result)
    print(edge_reply)
except Exception as e:
    print("Skipped because the Gemini API quota/rate limit was reached.")

=== STRUCTURED OUTPUT ===
{
  "location": "park entrance",
  "reported_issue": "People observed near park entrance, causing discomfort, potentially experiencing homelessness",
  "urgency": "LOW",
  "primary_responder": "Homeless Outreach Services",
  "supporting_services": [
    "Parks & Recreation"
  ],
  "resident_language": "English"
}

=== CIVIC ANALYSIS ===
--- PROBLEM ---
The report describes two individuals sitting near a park entrance with bags. There is no indication of obstruction, trash, or hazardous conditions. The concern is based on uncertainty and perceived discomfort rather than a clear civic issue.

--- IMPACT ---
There is minimal direct public health or safety impact based on the information provided. The situation may create a perception of discomfort, but no concrete risks or violations are described.

--- URGENCY ---
LOW — The situation does not present any immediate safety risk, obstruction, or environmental hazard. There is no evidence requiring urgent intervention.

--- ACTION ---
No immediate enforcement or cleanup response is necessary. A non-intrusive review or optional outreach could be considered, but no urgent action is required.

=== RESIDENT RESPONSE ===
Thank you for your report. We have received your concern regarding individuals near the park entrance. A city staff member will review the situation, and appropriate services may be considered if needed. For non-emergency issues, responses typically occur within 3–5 business days.

The prompt/output was acceptable as this situation is pretty vague making it hard to create a clear action and category alongside it not being as much of an emergency yet, AI was able to recognize this low urgency situation and the proper actions for it.